# Lecture 08 — Exercise solutions

These are model answers with executable checks. Alternative explanations are welcome when they state their assumptions.

## Notebook setup

The setup finds `data/churn.csv` from this notebook’s directory and adds the course source package. Model files and service checks use temporary paths.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "churn.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate data/churn.csv from the current directory")
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "churn.csv"
print("Project root:", ROOT)

## Exercise 1: Derive the threshold decision

At threshold 0.5, the labels are `[0, 0, 1, 1]`. At threshold 0.4, they are `[0, 1, 1, 1]`. The probability 0.43 changes decision because it crosses the cutoff. The fitted model still assigns the same probabilities; only the decision rule changes.

In [ ]:
probabilities = [0.18, 0.43, 0.51, 0.86]
labels_at_050 = [int(p >= 0.50) for p in probabilities]
labels_at_040 = [int(p >= 0.40) for p in probabilities]
assert labels_at_050 == [0, 0, 1, 1]
assert labels_at_040 == [0, 1, 1, 1]
print("threshold 0.50:", labels_at_050)
print("threshold 0.40:", labels_at_040)

## Exercise 2: A simple cost calculation

If we contact, expected cost is 1. If we do not contact, expected cost is \(4p\), where \(p\) is the probability of churn. Contact when \(1<4p\), or \(p>1/4\); the tie is at threshold 0.25. This assumes contact prevents churn with certainty, costs are fixed, and these are the only costs. If intervention effectiveness is \(q<1\), expected benefit is \(4pq\), so contact when \(4pq>1\).

In [ ]:
contact_cost = 1.0
missed_churn_cost = 4.0
tie_threshold = contact_cost / missed_churn_cost
assert tie_threshold == 0.25
print(f"Tie threshold: {tie_threshold:.2f}; contact when p > {tie_threshold:.2f}")

effectiveness = 0.75
effective_threshold = contact_cost / (missed_churn_cost * effectiveness)
print(f"At intervention effectiveness q={effectiveness:.2f}, contact when p > {effective_threshold:.3f}")

## Exercise 3: Keep the transformation with the estimator

A scaler fitted before splitting uses test-set means and variances, so training indirectly sees held-out distribution information. A pipeline fits the scaler and encoder only on its training rows, then applies those stored transformations at prediction time. The saved artifact includes the fitted pipeline.

In [ ]:
import joblib
import pandas as pd
import tempfile
from course_ml.train import FEATURES, train_model

with tempfile.TemporaryDirectory() as tmp:
    artifact_path = Path(tmp) / "churn.joblib"
    train_model(DATA, artifact_path)
    fitted = joblib.load(artifact_path)["pipeline"]
    probe = pd.DataFrame(
        [{"tenure_months": 8, "monthly_charge": 92.0, "support_calls": 4, "contract": "monthly"}],
        columns=FEATURES,
    )
    before_reload = fitted.predict_proba(probe)
    reloaded = joblib.load(artifact_path)["pipeline"]
    after_reload = reloaded.predict_proba(probe)
    assert (before_reload == after_reload).all()
    print("Probability before and after reload:", before_reload[0, 1])

## Exercise 4: Exercise the request boundary

With the model configured, a valid record returns 200. Unknown contract labels, negative tenure, missing required fields, and unexpected fields return 422. Rejecting extra fields catches misspelled or stale client fields instead of silently ignoring them.

In [ ]:
import os
from fastapi.testclient import TestClient
from course_ml import app as api
from course_ml.train import train_model

previous_present = "MODEL_PATH" in os.environ
previous_value = os.environ.get("MODEL_PATH")
try:
    with tempfile.TemporaryDirectory() as tmp:
        artifact_path = Path(tmp) / "churn.joblib"
        train_model(DATA, artifact_path)
        os.environ["MODEL_PATH"] = str(artifact_path)
        api._load_artifact.cache_clear()
        client = TestClient(api.app)
        valid = {"tenure_months": 8, "monthly_charge": 92.0, "support_calls": 4, "contract": "monthly"}
        cases = {
            "valid": (valid, 200),
            "unknown contract": ({**valid, "contract": "quarterly"}, 422),
            "negative tenure": ({**valid, "tenure_months": -1}, 422),
            "missing field": ({k: v for k, v in valid.items() if k != "contract"}, 422),
            "extra field": ({**valid, "admin": True}, 422),
        }
        results = {name: client.post("/predict", json=payload).status_code for name, (payload, _) in cases.items()}
        for name, status in results.items():
            print(f"{name}: HTTP {status}")
        assert all(results[name] == expected for name, (_, expected) in cases.items())
finally:
    api._load_artifact.cache_clear()
    if previous_present:
        os.environ["MODEL_PATH"] = previous_value
    else:
        os.environ.pop("MODEL_PATH", None)

## Exercise 5: Name a monitoring signal

Compare contract proportions in a reference training sample and an incoming batch. A large change can signal a customer-mix shift or a data-pipeline issue. This detects a distribution change; it does not prove that prediction quality fell. Outcome labels are needed to assess quality.

In [ ]:
data = pd.read_csv(DATA)
reference_contract_share = data["contract"].value_counts(normalize=True)

# Simulate an incoming batch with a changed contract mix for demonstration.
live_batch = data.sample(frac=0.20, random_state=17).copy()
live_batch["contract"] = "monthly"
live_contract_share = live_batch["contract"].value_counts(normalize=True)
monitor = pd.DataFrame({
    "reference_share": reference_contract_share,
    "live_share": live_contract_share,
}).fillna(0.0)
monitor["absolute_change"] = (monitor["live_share"] - monitor["reference_share"]).abs()
display(monitor.sort_values("absolute_change", ascending=False))